In [ ]:
import pandas as pd

data = "ca"
datapath = f"{data}/sample.csv"

df = pd.read_csv(datapath)
df["UTCTimeOffset"] = pd.to_datetime(df["UTCTimeOffset"])


df = df.sort_values(["UserId", "UTCTimeOffset"]).reset_index(drop=True)
seq_cols = [
    "PoiId",
    "PoiCategoryId",
    "PoiCategoryCode",
    "PoiCategoryName",
    "Latitude",
    "Longitude",
    "UTCTimeOffset",
    "pseudo_session_trajectory_id"
    # "UserCheckinRank"
]


train_df = df[df["SplitTag"] == "train"].copy()
train_users = set(train_df["UserId"].unique())
train_pois = set(train_df["PoiId"].unique())
df = df[df["UserId"].isin(train_users) & df["PoiId"].isin(train_pois)].copy()

df = df.sort_values(["UserId", "UTCTimeOffset"]).reset_index(drop=True)

print(df["SplitTag"].value_counts())

results = []

for user_id, user_df in df.groupby("UserId", sort=False):
    user_df = user_df.sort_values("UTCTimeOffset").reset_index(drop=True)

    for (traj_id, split_tag), traj_df in user_df.groupby(["pseudo_session_trajectory_id", "SplitTag"], sort=False):
        traj_df = traj_df.sort_values("UTCTimeOffset").reset_index(drop=True)
        split_tag = traj_df["SplitTag"].iloc[0]
        start_time = traj_df["UTCTimeOffset"].iloc[0]
        history_df = user_df[user_df["UTCTimeOffset"] < start_time].copy()
        current_df = traj_df.copy()
        merged_df = pd.concat([history_df, current_df], axis=0).sort_values("UTCTimeOffset")
        if len(merged_df) > 50:
            merged_df = merged_df.iloc[-50:].reset_index(drop=True)
        else:
            merged_df = merged_df.reset_index(drop=True)

        if split_tag == "train" and len(merged_df) < 20:
            continue

        record = {
            "UserId": user_id,
            "trajectory_id": traj_id,
            "SplitTag": split_tag,
            "history_count": len(history_df),
            "current_count": len(current_df),
            "final_count": len(merged_df),
            "sequence_PoiId": merged_df["PoiId"].tolist(),
            "sequence_PoiCategoryId": merged_df["PoiCategoryId"].tolist(),
            "sequence_PoiCategoryName": merged_df["PoiCategoryName"].tolist(),
            "sequence_Latitude": merged_df["Latitude"].tolist(),
            "sequence_Longitude": merged_df["Longitude"].tolist(),
            "sequence_UTCTimeOffset": merged_df["UTCTimeOffset"].astype(str).tolist(),
            "sequence_pseudo_session_trajectory_id": merged_df["pseudo_session_trajectory_id"].tolist(),
            # "sequence_UserCheckinRank": merged_df["UserCheckinRank"].tolist(),
        }

        results.append(record)

result_df = pd.DataFrame(results)

# print(result_df.head(10))

result_df.to_csv(f"{data}/trajectory_sequence_data.csv", index=False)

In [ ]:
train_df = result_df[result_df["SplitTag"] == "train"][["UserId", "sequence_PoiId", "sequence_UTCTimeOffset"]]
val_df = result_df[result_df["SplitTag"] == "validation"][["UserId", "sequence_PoiId", "sequence_UTCTimeOffset"]]
test_df = result_df[result_df["SplitTag"] == "test"][["UserId", "sequence_PoiId", "sequence_UTCTimeOffset"]]

train_df.to_csv(f"{data}/new/train_poi_sequence.csv", index=False)
val_df.to_csv(f"{data}/new/validation_poi_sequence.csv", index=False)
test_df.to_csv(f"{data}/new/test_poi_sequence.csv", index=False)

print("train:", len(train_df))
print("validation:", len(val_df))
print("test:", len(test_df))

train: 22139
validation: 3529
test: 2780


In [ ]:
import pandas as pd 
data = "ca" 
datapath = f"{data}/test_sample.csv" 

df = pd.read_csv(datapath)
trajectory_id = df["pseudo_session_trajectory_id"].unique()
print(len(trajectory_id))


2780


### POI info

In [ ]:
import pandas as pd
import os.path as osp
import ast

def extract_category_name(x, data):
    if pd.isna(x):
        return None
    if data == "ca":
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list) and len(parsed) > 0 and isinstance(parsed[0], dict):
                return parsed[0].get("name", None)
            return str(x)
        except Exception:
            return str(x)
    return str(x)


def poi_info(data):
    datapath = f"{data}/train_sample.csv"
    output_path = f"{data}/poi_info.csv"

    df = pd.read_csv(datapath)
    df = df[["PoiId", "PoiCategoryName", "Latitude", "Longitude", "UTCTimeOffset"]].copy()
    df["UTCTimeOffset"] = pd.to_datetime(df["UTCTimeOffset"], errors="coerce")
    df["PoiCategoryName"] = df["PoiCategoryName"].apply(lambda x: extract_category_name(x, data))

    poi_info_data = []

    for pid, group in df.groupby("PoiId"):
        group = group.dropna(subset=["UTCTimeOffset"])
        if len(group) == 0:
            continue

        row0 = group.iloc[0]
        category = row0["PoiCategoryName"]
        latitude = row0["Latitude"]
        longitude = row0["Longitude"]

        hours = group["UTCTimeOffset"].dt.hour
        hour_counts = hours.value_counts().to_dict()

        count_threshold = 1
        filtered_hour_counts = {
            int(h): int(c) for h, c in hour_counts.items() if c >= count_threshold
        }

        sorted_hour_counts = dict(
            sorted(filtered_hour_counts.items(), key=lambda x: x[1], reverse=True)
        )

        poi_info_data.append({
            "PoiId": pid,
            "PoiCategoryName": category,
            "Latitude": latitude,
            "Longitude": longitude,
            "visit_time_and_count": sorted_hour_counts
        })

    poi_info_df = pd.DataFrame(poi_info_data)
    poi_info_df.to_csv(output_path, index=False)

    print(f"{output_path}，{len(poi_info_df)} POI")


data = "ca"
poi_info(data)

